# Study 864 — Yield-Curve Twist (Butterfly) 🔀

**Beyond level and slope: does the curve's *curvature* predict returns?**

The Treasury curve moves in three modes (Litterman & Scheinkman 1991): a **level** shift, a
**slope** steepening, and a **curvature** — a *butterfly*, the belly moving relative to the two
wings. We build the 5-10-30 butterfly `fly = 2·y10 − y5 − y30` from `^FVX`/`^TNX`/`^TYX` and ask
whether it (and its *change*, a "twist") predicts forward IEF / TLT / SPY returns — *distinct* from
the 2s10s slope (studies 66/132) and roll-down carry (380). Tape: 2002-07-31 → 2026-06-30,
6,011 days (fingerprint `5136638af800`).

*Numbers below are the frozen headline (`docs/results.md`); the live cells run the fast synthetic
control. No cross-section → no survivorship bias.*


## 1. What is a butterfly?

Draw a straight line from the **5-year** yield to the **30-year** yield. Where does the **10-year** (the *belly*) sit relative to that line? The butterfly measures exactly that gap:

$$\text{fly} = 2\,y_{10} - y_5 - y_{30}$$

A **positive** fly means the belly yield sits *above* the line — the 10-year is **cheap** (its price is low). The folklore: a cheap belly mean-reverts, so a high fly should precede belly bonds (IEF, the 7-10y ETF) *rising*. A *twist* is a day-over-day **change** in that curvature.

In [1]:
R = {'start': '2002-07-31', 'end': '2026-06-30', 'n': 6011, 'fp': '5136638af800', 'ief5_t': 2.13, 'ief5_b': 3.6, 'ief21_t': 2.34, 'ief21_b': 16.9, 'ief21_r2': 1.58, 'ief63_t': 2.42, 'ief63_b': 46.8, 'ief63_r2': 4.04, 'tlt21_t': 1.12, 'tlt21_b': 16.1, 'spy21_t': -1.6, 'spy21_b': -24.7, 'inc_fly_b': 20.2, 'inc_fly_t': 1.98, 'inc_slope_t': 0.25, 'inc_level_t': -0.92, 'q5': 58.5, 'q1': 5.4, 'q_spread': 53.0, 'q_t': 1.66, 'dfly5_t': 0.65, 'dfly21_t': -0.38, 'dfly63_t': 1.56, 'era1_b': 48.7, 'era1_t': 3.32, 'era1_r2': 10.32, 'era1_n': 1782, 'era2_b': -1.8, 'era2_t': -0.18, 'era2_n': 1927, 'era3_b': 7.2, 'era3_t': 0.59, 'era3_n': 2050, 'plac_obs_t': 2.337, 'plac_sd': 1.047, 'plac_p': 0.024, 'plac_n': 500, 'timer1_active': 1.02, 'timer1_passive': 1.41, 'timer1_spread': -0.39, 'timer1_t': -0.96, 'timer1_sh_a': 0.557, 'timer1_sh_p': 0.522, 'timer5_spread': -0.61, 'timer5_sh_a': 0.439, 'sw_yr': 13.5, 'plant_t': 33.99, 'plant_spread_t': 24.7, 'null_mean_t': 0.03, 'null_sd_t': 1.37, 'null_fire': 5, 'null_seeds': 20}
print('butterfly -> forward IEF (belly Treasury) return, HAC t:')
print(f"  5d : beta {R['ief5_b']:+.1f} bps/1sigma   t = {R['ief5_t']:+.2f}")
print(f"  21d: beta {R['ief21_b']:+.1f} bps/1sigma  t = {R['ief21_t']:+.2f}  (R2 {R['ief21_r2']:.2f}%)")
print(f"  63d: beta {R['ief63_b']:+.1f} bps/1sigma  t = {R['ief63_t']:+.2f}  (R2 {R['ief63_r2']:.2f}%)")
print('  -> right sign (cheap belly -> belly rallies), full-sample t just past 2.')

butterfly -> forward IEF (belly Treasury) return, HAC t:
  5d : beta +3.6 bps/1sigma   t = +2.13
  21d: beta +16.9 bps/1sigma  t = +2.34  (R2 1.58%)
  63d: beta +46.8 bps/1sigma  t = +2.42  (R2 4.04%)
  -> right sign (cheap belly -> belly rallies), full-sample t just past 2.


## 2. But is it stable? The decisive era cut

A full-sample *t* just past 2 is fragile. Split the tape into three eras and the story collapses — the **entire** effect is a pre-2010 relic:

In [2]:
print('fly -> IEF (h=21), by era:')
print(f"  2002-2009: beta {R['era1_b']:+.1f}  t = {R['era1_t']:+.2f}  (R2 {R['era1_r2']:.1f}%)  <- all the juice")
print(f"  2010-2017: beta {R['era2_b']:+.1f}  t = {R['era2_t']:+.2f}  <- dead")
print(f"  2018-2026: beta {R['era3_b']:+.1f}  t = {R['era3_t']:+.2f}  <- dead")

fly -> IEF (h=21), by era:
  2002-2009: beta +48.7  t = +3.32  (R2 10.3%)  <- all the juice
  2010-2017: beta -1.8  t = -0.18  <- dead
  2018-2026: beta +7.2  t = +0.59  <- dead


## 3. A live synthetic control — is the machinery honest?

We plant a butterfly→return edge in a seeded toy world (`fly_signal>0`) and check the detector recovers it — and stays *silent* on the null (`fly_signal=0`, curvature wanders but predicts nothing). No network.

In [3]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from curve_twist import data, strategy as st
planted = st.synthetic_detect(data.synthetic_daily(n_days=3000, fly_signal=0.02, seed=864)[0], 21)
null    = st.synthetic_detect(data.synthetic_daily(n_days=3000, fly_signal=0.0,  seed=864)[0], 21)
print('planted world: regression t = %+.2f  (should light up)' % planted['t'])
print('null world   : regression t = %+.2f  (should be ~0)'    % null['t'])

planted world: regression t = +29.15  (should light up)
null world   : regression t = +0.58  (should be ~0)


## 4. The honest verdict

The butterfly **does** predict forward belly-Treasury returns the right way (NW *t* = **+2.34** at 21d), and it isn't just the 2s10s slope in disguise (the slope control is insignificant). **But** the whole effect is a **2002-2009** relic (*t* = +3.32) that dies in 2010-2017 (*t* = -0.18) and 2018-2026 (*t* = +0.59); the *twist* (the change) predicts nothing; and a curvature timer **loses to buy-and-hold** after costs. **Signal: Weak** (right sign, unstable magnitude), **Tradability: Mirage**.